# 03 - 均线交叉策略回测

本 Notebook 演示：
1. 实现均线金叉/死叉策略
2. 使用 backtrader 进行回测
3. 分析回测绩效指标
4. 对比买入持有策略

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import backtrader as bt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

from src.data.storage import load_from_parquet
from src.indicators.calculator import add_all_indicators
from src.strategy.ma_crossover import MACrossoverStrategy, MACrossoverBT
from src.backtest.engine import BacktestEngine

print('Modules loaded')

## 1. Pandas 向量化策略验证

In [ ]:
# 加载数据并生成信号
df = load_from_parquet('600036')
strategy = MACrossoverStrategy(short_period=5, long_period=20)
signals = strategy.generate_signals(df)

print(f'Buy signals: {(signals["signal"]==1).sum()}')
print(f'Sell signals: {(signals["signal"]==-1).sum()}')

# 可视化最近信号
fig, ax = plt.subplots(figsize=(14, 6))
subset = signals.tail(200)
ax.plot(subset['date'], subset['close'], color='black', linewidth=0.8, label='Close')
ax.plot(subset['date'], subset['ma_5'], color='blue', linewidth=0.8, alpha=0.6, label='MA5')
ax.plot(subset['date'], subset['ma_20'], color='purple', linewidth=0.8, alpha=0.6, label='MA20')

# 标记买卖信号
buys = subset[subset['signal'] == 1]
sells = subset[subset['signal'] == -1]
ax.scatter(buys['date'], buys['close'], color='red', s=50, marker='^', zorder=5, label='Buy')
ax.scatter(sells['date'], sells['close'], color='green', s=50, marker='v', zorder=5, label='Sell')

ax.set_title('MA Crossover Signals (600036)')
ax.set_ylabel('Price (CNY)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.show()

## 2. Backtrader 完整回测

In [ ]:
# 准备数据
bt_data = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()

# 创建回测引擎
engine = BacktestEngine(initial_cash=10000.0)
engine.add_data(bt_data, symbol='600036')
engine.add_strategy(MACrossoverBT, short_period=5, long_period=20)

# 运行回测
results = engine.run()
engine.print_summary()

## 3. 策略理解要点

### 金叉 (Golden Cross) 和 死叉 (Death Cross)
- **金叉**: 短期均线从下方向上穿过长期均线 → 视为上涨趋势开始 → **买入信号**
- **死叉**: 短期均线从上方向下穿过长期均线 → 视为下跌趋势开始 → **卖出信号**

### 参数含义
- `short_period=5`: 5日移动平均线（对价格变化更敏感）
- `long_period=20`: 20日移动平均线（更平滑，反映中期趋势）

### 为什么胜率只有 36% 却盈利 33%？
- 这是因为「截断亏损，让利润奔跑」(Cut Losses, Let Profits Run)
- 亏损的交易亏损较少（均线信号反应相对及时）
- 盈利的交易盈利较多（捕捉到大的趋势行情）
- **盈亏比** > 1 是盈利的关键，胜率反而不是最重要的

### 策略局限性
- 震荡市中频繁产生假信号（频繁金叉死叉）
- 单边上涨不如 Buy & Hold
- 在大跌市中可能反复进出亏损